In [ ]:
#  CONFIGURAÇÃO INICIAL E IMPORTAÇÕES

# Preparar o ambiente para rodar Apache Spark junto com Python, além de instalar bibliotecas
# auxiliares para análise de dados, machine learning e visualização interativa.

# Instalações necessárias
# O openjdk-8-jdk-headless já está na imagem base do Dockerfile.
# As bibliotecas pip serão instaladas pelo Dockerfile.
# !apt-get install -y openjdk-8-jdk-headless > /dev/null # Removido, já na imagem base
# !pip install -q findspark pyspark==3.5.0 folium seaborn plotly matplotlib ipywidgets scikit-learn # Removido, já no Dockerfile

# Configuração do ambiente
import os
import findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
findspark.init()

# Importações principais
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, year, month, dayofmonth, count, desc, sum as _sum, avg, monotonically_increasing_id
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, PCA, Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, NaiveBayes
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


# Visualização e utilitários
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import folium
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
import ipywidgets as widgets
from IPython.display import display, clear_output
import pickle

# INICIALIZAÇÃO DO SPARK E CARREGAMENTO DOS DADOS

# O Spark foi escolhido por sua capacidade de processamento distribuído, essencial para
# análise de grandes volumes de dados. A versão 3.5.0 oferece estabilidade e compatibilidade
# com todas as bibliotecas que utilizaremos.

# Criando e inicializando a Spark Session, que é a porta de entrada para trabalhar com o Spark no Python.
# É através dela que conseguimos ler dados, processar, fazer transformações, aplicar machine learning e muito mais.

# Criação da Spark Session
spark = SparkSession.builder \
    .appName("Analise de IST") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Apache Spark configurado e iniciado!")
print("Versão do Spark:", spark.version)

# Primeiro, montamos o Google Drive para conseguir acessar o arquivo CSV que contém os dados que serão utilizados na análise.
# from google.colab import drive
# drive.mount('/content/drive') # Removido, não é necessário em Docker

# Definimos o caminho do arquivo CSV dentro do Google Drive.
# É importante que o caminho esteja correto para evitar erros na leitura.
#file_path = '/content/drive/MyDrive/dados_ist_tratados.csv' # Caminho original do Colab
file_path = 'data/dados_ist_tratados.csv' # Novo caminho no container Docker

# Utilizamos o Spark para fazer a leitura do arquivo CSV.
# Colocamos a opção header=True para que a primeira linha do arquivo seja reconhecida como cabeçalho (nomes das colunas).
# E inferSchema=True para que o Spark identifique automaticamente os tipos de dados de cada coluna.
spark_df = spark.read.csv(file_path, header=True, inferSchema=True)

# Visualização inicial
print("\nSchema do DataFrame:")
spark_df.printSchema()
print("\nPrimeiras 5 linhas:")
spark_df.show(5)

# PRÉ-PROCESSAMENTO DOS DADOS
# O pré-processamento é essencial para garantir a qualidade dos dados antes da modelagem.
# Inclui tratamento de valores ausentes, codificação de variáveis categóricas e criação
# de features derivadas que podem ser úteis para os modelos.

# Listas para classificação de doenças
ists = ["HIV", "Sífilis", "Gonorreia", "HPV", "Clamídia", "Herpes Genital"]
curaveis = ["Sífilis", "Gonorreia", "Clamídia"]

# Criação de colunas binárias:
# - 'tem_ist': identifica se o indivíduo possui alguma IST listada (1 para sim, 0 para não).
# - 'curavel': identifica se a IST é considerada curável (1 para sim, 0 para não).
# Isso permite realizar análises e modelagens considerando a presença de ISTs e sua possibilidade de cura.
spark_df = spark_df.withColumn("tem_ist", when(col("doenca").isin(ists), 1).otherwise(0))
spark_df = spark_df.withColumn("curavel", when(col("doenca").isin(curaveis), 1).otherwise(0))

## Indexadores para colunas categóricas
#Etapa 1: Indexação de colunas categóricas
# O StringIndexer transforma categorias em números inteiros.
# O parâmetro handleInvalid='keep' garante que, se houver valores não reconhecidos, eles não causarão erro e serão tratados.

indexers = [
    StringIndexer(inputCol="genero", outputCol="generoIdx", handleInvalid="keep"),
    StringIndexer(inputCol="doenca", outputCol="doencaIdx", handleInvalid="keep"),
    StringIndexer(inputCol="localidade", outputCol="localidadeIdx", handleInvalid="keep"),
    StringIndexer(inputCol="nivel_educacional", outputCol="nivelEducacionalIdx", handleInvalid="keep")
]

## OneHotEncoder para transformar os indexadores
# Etapa 2: OneHotEncoder para variáveis categóricas
# O OneHotEncoder cria vetores binários para cada categoria,
# evitando que o modelo interprete uma ordem inexistente entre categorias.

encoders = [
    OneHotEncoder(inputCol="generoIdx", outputCol="generoVec"),
    OneHotEncoder(inputCol="doencaIdx", outputCol="doencaVec"),
    OneHotEncoder(inputCol="localidadeIdx", outputCol="localidadeVec"),
    OneHotEncoder(inputCol="nivelEducacionalIdx", outputCol="nivelEducacionalVec")
]

## Montagem do vetor de features
## Etapa 3: Montagem do vetor de features
# O VectorAssembler reúne todas as variáveis numéricas e categóricas codificadas
# em uma única coluna chamada 'features', que será utilizada pelos modelos.
assembler = VectorAssembler(
    inputCols=["idade", "renda_media", "generoVec", "doencaVec", "localidadeVec", "nivelEducacionalVec"],
    outputCol="features"
)

# Pipeline completo
# Etapa 4: Criação do pipeline de transformação
# O pipeline automatiza as etapas de indexação, codificação e montagem do vetor de features.

pipeline = Pipeline(stages=indexers + encoders + [assembler])

# Aplicar pipeline no dataframe original
model = pipeline.fit(spark_df)
encoded_df = model.transform(spark_df)

# Criar coluna binária 'tem_ist': 1 se é uma IST, 0 se não
# Etapa 5: Criação da coluna binária 'tem_ist'
# Criamos uma variável alvo que indica se a pessoa possui ou não uma IST.
# A condição é: se a coluna 'doenca' for diferente de "Nenhuma", então tem IST (1), senão não tem (0).

encoded_df = encoded_df.withColumn("tem_ist", when(col("doenca").isin(ists), 1).otherwise(0))

# Criar coluna binária 'curavel': 1 se é curável, 0 se não (só nas ISTs)
encoded_df = encoded_df.withColumn("curavel", when(col("doenca").isin(curaveis), 1).otherwise(0))

# Conferimos as primeiras linhas após o pré-processamento para garantir que tudo foi aplicado corretamente.
encoded_df.select("id", "idade", "renda_media", "genero", "localidade", "doenca", "tem_ist", "curavel").show(5)

# PROCESSAMENTO DE LINGUAGEM NATURAL (PLN)
# Aplicamos técnicas de PLN na coluna 'doenca' para extrair informações adicionais
# dos textos. O TF-IDF é uma técnica eficaz para representar textos de forma numérica,
# permitindo análises quantitativas de conteúdo textual.

# Pipeline para Processamento de Linguagem Natural (PLN) na coluna 'doenca'.
# Esse pipeline converte os dados textuais em vetores numéricos que podem ser utilizados em modelos de Machine Learning.

# Ao final, gera uma nova coluna chamada 'textFeatures' que contém a representação vetorial dos textos,
# permitindo que esses dados sejam utilizados em modelos de Machine Learning.

tokenizer = Tokenizer(inputCol="doenca", outputCol="palavras")
remover = StopWordsRemover(inputCol="palavras", outputCol="palavras_filtradas")
hashing_tf = HashingTF(inputCol="palavras_filtradas", outputCol="rawFeatures", numFeatures=100)
idf = IDF(inputCol="rawFeatures", outputCol="textFeatures")

pln_pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])
pln_model = pln_pipeline.fit(spark_df)
tfidf_data = pln_model.transform(spark_df)

# Visualização dos resultados do TF-IDF
print("\nResultados do TF-IDF:")
tfidf_data.select("doenca", "textFeatures").show(5, truncate=False)

# DATA WAREHOUSE E OLAP
# O Data Warehouse permite análises multidimensionais (OLAP) dos dados. Criamos dimensões
# como Tempo, Localidade, Doença, etc., que permitem "fatiar" os dados de diferentes formas
# para responder a diversas perguntas de negócio.

#Fato: Tabela que armazena os eventos — neste caso, os testes ou registros de IST,
#contendo informações como idade, renda, doença, localidade e data do teste.
#Dimensões:
#Tempo: Quebra a data em ano, mês e dia para permitir análises temporais.
#Localidade: Permite entender a distribuição geográfica dos casos.
#Doença: Permite agrupar por tipo de IST e também classificar como curável ou não.
#Escolaridade: Para entender o impacto do nível educacional na incidência das doenças.
#Gênero: Permite recortes por gênero para análises demográficas.

# Dimensão Tempo
dim_tempo = spark_df.select("data_teste").dropDuplicates()
dim_tempo = dim_tempo.withColumn("ano", year("data_teste")) \
                     .withColumn("mes", month("data_teste")) \
                     .withColumn("dia", dayofmonth("data_teste"))

# Dimensão Localidade
dim_localidade = spark_df.select("localidade").dropDuplicates()

# Dimensão Doença
dim_doenca = spark_df.select("doenca").dropDuplicates() \
                      .withColumn("curavel", when(col("doenca").isin(curaveis), "Sim") \
                      .otherwise("Não"))

# Dimensão Escolaridade
dim_escolaridade = spark_df.select("nivel_educacional").dropDuplicates()

# Dimensão Gênero
dim_genero = spark_df.select("genero").dropDuplicates()

# Tabela de Fato - Casos
fato_casos = spark_df.select(
    "id", "idade", "renda_media", "data_teste", "localidade",
    "doenca", "nivel_educacional", "genero", "tem_ist", "curavel"
)

print("\nTabelas de dimensões e fato criadas com sucesso!")

# Registro das tabelas temporárias no Spark para consultas SQL.
# Isso permite que os DataFrames sejam consultados diretamente utilizando comandos SQL dentro do ambiente Spark.

fato_casos.createOrReplaceTempView("fato_casos")
dim_tempo.createOrReplaceTempView("dim_tempo")
dim_localidade.createOrReplaceTempView("dim_localidade")
dim_doenca.createOrReplaceTempView("dim_doenca")
dim_escolaridade.createOrReplaceTempView("dim_escolaridade")
dim_genero.createOrReplaceTempView("dim_genero")

# Criação de um menu interativo (Dropdown) para realizar análises OLAP dinâmicas.

# O Dropdown permite que o usuário escolha qual análise deseja visualizar entre as opções:
# - Casos por Localidade
# - Média de Idade por Doença
# - Média de Renda por Escolaridade e Doença
# - Casos por Ano

# Ao clicar no botão 'Executar Análise', uma função é chamada para rodar a consulta SQL correspondente
# sobre os dados da Data Warehouse, utilizando Spark SQL.

# O resultado de cada análise é convertido para Pandas e, em seguida, apresentado graficamente com Seaborn e Matplotlib.

# Isso permite que o usuário explore os dados de forma interativa, visualizando os resultados das análises OLAP
# de maneira clara, intuitiva e com gráficos de alta qualidade.

# Dropdown para escolher a análise OLAP
# Este widget é interativo e não será executado de forma significativa por nbconvert.
# Para automação, você pode remover este bloco ou modificar a função para executar todas as análises.
# analise_dropdown = widgets.Dropdown(
#     options=[
#         'Casos por Localidade',
#         'Média de Idade por Doença',
#         'Média de Renda por Escolaridade e Doença',
#         'Casos por Ano'
#     ],
#     description='Escolha a Análise:',
#     style={'description_width': 'initial'},
#     layout=widgets.Layout(width='60%')
# )

# # Botão para executar
# botao_executar = widgets.Button(description="Executar Análise", button_style='success')

# # Área de saída
# saida = widgets.Output()

# Função da OLAP
# Esta função precisa ser chamada diretamente para cada análise que você deseja executar
# se os widgets forem removidos.
def executar_analise_automatica(escolha):
    # with saida: # Removido, pois não há widget de saída
        # clear_output() # Removido, pois não há widget de saída

        # Transformar a lista de ISTs em string para a query SQL
        ist_str = ', '.join([f"'{d}'" for d in ists])

        tons_azuis = sns.color_palette("Blues", n_colors=6)

        if escolha == 'Casos por Localidade':
            resultado = spark.sql(f"""
                SELECT localidade, COUNT(*) AS total_casos
                FROM fato_casos
                WHERE doenca IN ({ist_str})
                GROUP BY localidade
                ORDER BY total_casos DESC
            """).toPandas()

            plt.figure(figsize=(18,9))
            sns.barplot(data=resultado, x="localidade", y="total_casos", color=tons_azuis[4])
            plt.title("Casos por Localidade (Somente ISTs)")
            plt.xlabel("Localidade")
            plt.ylabel("Número de Casos")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

        elif escolha == 'Média de Idade por Doença':
            resultado = spark.sql(f"""
                SELECT doenca, ROUND(AVG(idade), 2) AS media_idade
                FROM fato_casos
                WHERE doenca IN ({ist_str})
                GROUP BY doenca
                ORDER BY media_idade DESC
            """).toPandas()

            plt.figure(figsize=(10,6))
            sns.barplot(data=resultado, x="doenca", y="media_idade", palette="Blues")
            plt.title("Média de Idade por Tipo de IST")
            plt.xlabel("Doença")
            plt.ylabel("Média de Idade")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

        elif escolha == 'Média de Renda por Escolaridade e Doença':
            resultado = spark.sql(f"""
                SELECT nivel_educacional, doenca, ROUND(AVG(renda_media), 2) AS media_renda
                FROM fato_casos
                WHERE doenca IN ({ist_str})
                GROUP BY nivel_educacional, doenca
                ORDER BY media_renda DESC
            """).toPandas()

            plt.figure(figsize=(15,6))
            sns.barplot(data=resultado, x="nivel_educacional", y="media_renda", hue="doenca", palette="Blues")
            plt.title("Média de Renda por Escolaridade e Doença (Somente ISTs)")
            plt.xlabel("Nível Educacional")
            plt.ylabel("Média de Renda")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

        elif escolha == 'Casos por Ano':
            resultado = spark.sql(f"""
                SELECT YEAR(data_teste) AS ano_teste, COUNT(*) AS total_casos
                FROM fato_casos
                WHERE doenca IN ({ist_str})
                GROUP BY ano_teste
                ORDER BY ano_teste
            """).toPandas()

            plt.figure(figsize=(10,6))
            sns.lineplot(data=resultado, x="ano_teste", y="total_casos", marker="o", color=tons_azuis[4])
            plt.title("Número de Casos por Ano (Somente ISTs)")
            plt.xlabel("Ano")
            plt.ylabel("Total de Casos")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

# Conectar botão à função # Removido, pois não há widget de botão
# botao_executar.on_click(executar_analise)

# Exibir os widgets # Removido, pois não há widgets interativos
# display(analise_dropdown, botao_executar, saida)

# Chame a função para cada análise que você deseja executar automaticamente
executar_analise_automatica('Casos por Localidade')
executar_analise_automatica('Média de Idade por Doença')
executar_analise_automatica('Média de Renda por Escolaridade e Doença')
executar_analise_automatica('Casos por Ano')


# Nesta etapa, realizamos uma análise utilizando o paradigma Hadoop MapReduce aplicado no Spark.

# Objetivo: calcular a MÉDIA DE RENDA por tipo de IST.

# Primeiro, os dados são convertidos para um RDD, que é a estrutura utilizada pelo Spark para processamento no estilo MapReduce.
# Depois, aplicamos:

# - Map: cada registro é transformado em uma tupla (doença, (renda, 1)), onde '1' representa a contagem.
# - Reduce: somamos as rendas e as contagens para cada doença.
# - MapValues: calculamos a média dividindo a soma total da renda pelo número de ocorrências de cada doença.

# O resultado é convertido novamente para um DataFrame para facilitar a visualização e geração de um gráfico de barras.

# Este processo simula uma análise no estilo Hadoop MapReduce, demonstrando como esse paradigma funciona para agregações simples,
# sendo especialmente útil para trabalhar com grandes volumes de dados distribuídos.

# Média de renda por tipo de IST (MapReduce)

# Filtrar apenas ISTs
ist_rdd = spark_df.filter(col("doenca").isin(ists)).rdd

# MapReduce - Média de renda por tipo de IST
map_renda = ist_rdd.map(lambda row: (row.doenca, (row.renda_media, 1)))
reduce_renda = map_renda.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
media_renda = reduce_renda.mapValues(lambda v: round(v[0]/v[1], 2)).toDF(["Doenca", "Media_Renda"])

media_renda.show()

# Gráfico
media_renda_pd = media_renda.toPandas()

plt.figure(figsize=(10,6))
sns.barplot(data=media_renda_pd, x="Doenca", y="Media_Renda", palette="Blues")
plt.title("Média de Renda por Tipo de IST - Análise Hadoop (MapReduce)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Nesta análise, utilizamos o paradigma Hadoop MapReduce para calcular a DISTRIBUIÇÃO DE CASOS POR FAIXA ETÁRIA.

# Primeiramente, definimos uma função chamada 'faixa_etaria' que categoriza as idades em grupos:
# - 0-17, 18-24, 25-34, 35-44, 45-59 e 60+.

# Em seguida, aplicamos:
# - Map: cada registro é transformado em uma tupla (faixa_etaria, 1), onde '1' representa a ocorrência de um caso.
# - Reduce: somamos as ocorrências para cada faixa etária.

# O resultado é convertido novamente para um DataFrame para facilitar a visualização dos dados.

# Por fim, geramos um gráfico de barras que exibe claramente como os casos estão distribuídos entre as diferentes faixas etárias.

# Este processo demonstra o uso do MapReduce no Spark para sumarização de dados categóricos, sendo muito eficiente
# para análise de grandes volumes de dados distribuídos.


#Distribuição de casos por faixa etária (MapReduce)

# Criar faixas etárias
def faixa_etaria(idade):
    if idade < 18:
        return "0-17"
    elif idade < 25:
        return "18-24"
    elif idade < 35:
        return "25-34"
    elif idade < 45:
        return "35-44"
    elif idade < 60:
        return "45-59"
    else:
        return "60+"

# Aplicar MapReduce
faixa_rdd = spark_df.rdd.map(lambda row: (faixa_etaria(row.idade), 1))
resultado_faixa = faixa_rdd.reduceByKey(lambda a, b: a + b).toDF(["Faixa_Etaria", "Total_de_Casos"])

resultado_faixa.show()

# Gráfico
resultado_faixa_pd = resultado_faixa.toPandas()

plt.figure(figsize=(8,6))
sns.barplot(data=resultado_faixa_pd, x="Faixa_Etaria", y="Total_de_Casos", palette="Blues")
plt.title("Distribuição de Casos por Faixa Etária - Análise Hadoop (MapReduce)")
plt.tight_layout()
plt.show()

##ANÁLISE E GRÁFICOS BÁSICOS

# Nesta análise, extraímos o ano da coluna 'data_teste' e criamos uma nova coluna chamada 'ano_teste'.

# A seguir, realizamos uma agregação que conta o número de casos para cada ano presente na base de dados,
# utilizando a função groupBy junto com a função de contagem (count).

# Esse agrupamento nos permite visualizar a evolução dos casos de IST ao longo dos anos.

# Por fim, os dados são convertidos para um DataFrame Pandas e plotamos um gráfico de barras,
# onde é possível observar, de forma clara e intuitiva, como os casos de IST se distribuíram ano a ano.

# Este tipo de análise temporal é essencial para entender tendências, identificar picos ou quedas nos registros
# e pode auxiliar na formulação de políticas públicas ou estratégias de prevenção.


# Número de Casos por Ano
spark_df = spark_df.withColumn("ano_teste", year("data_teste"))
casos_por_ano = spark_df.groupBy("ano_teste").agg(count("id").alias("num_casos"))
casos_por_ano_pd = casos_por_ano.toPandas()

plt.figure(figsize=(10,6))
plt.bar(casos_por_ano_pd["ano_teste"], casos_por_ano_pd["num_casos"], color='lightblue')
plt.xlabel("Ano")
plt.ylabel("Número de Casos")
plt.title("Número de Casos de IST por Ano")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Casos por Localidade no Mapa com Folium

# Criamos um mapa interativo para mostrar a distribuição dos casos por localidade.
# Cada marcador representa uma cidade, com tamanho e cor proporcionais à quantidade de casos.
# Isso ajuda a visualizar rapidamente quais regiões têm maior incidência de ISTs.

coords = {
    "Manaus": [-3.1019, -60.025],
    "Belém": [-1.4558, -48.5044],
    "Porto Velho": [-8.7619, -63.9039],
    "Rio Branco": [-9.9747, -67.8100],
    "Macapá": [0.0356, -51.0705],
    "Boa Vista": [2.8200, -60.6720],
    "Santarém": [-2.4385, -54.6996],
    "Palmas": [-10.1675, -48.3277],
    "Salvador": [-12.9747, -38.4767],
    "Fortaleza": [-3.7167, -38.5500],
    "Recife": [-8.0500, -34.9000],
    "São Luís": [-2.5297, -44.3044],
    "Maceió": [-9.6658, -35.7333],
    "Natal": [-5.8128, -35.2551],
    "João Pessoa": [-7.1200, -34.8800],
    "Teresina": [-5.0892, -42.8016],
    "Aracaju": [-10.9472, -37.0731],
    "Feira de Santana": [-12.2667, -38.9667],
    "Brasília": [-15.7939, -47.8828],
    "Goiânia": [-16.6869, -49.2648],
    "Campo Grande": [-20.4697, -54.6201],
    "Cuiabá": [-15.6014, -56.0979],
    "Anápolis": [-16.3281, -48.9528],
    "Dourados": [-22.2231, -54.8122],
    "Rio Verde": [-17.7923, -50.9192],
    "São Paulo": [-23.5505, -46.6333],
    "Rio de Janeiro": [-22.9068, -43.1729],
    "Belo Horizonte": [-19.9167, -43.9345],
    "Vitória": [-20.3155, -40.3128],
    "Campinas": [-22.9099, -47.0626],
    "São José dos Campos": [-23.1896, -45.8841],
    "Ribeirão Preto": [-21.1775, -47.8103],
    "Uberlândia": [-18.9141, -48.2749],
    "Curitiba": [-25.4284, -49.2733],
    "Porto Alegre": [-30.0346, -51.2177],
    "Florianópolis": [-27.5954, -48.5480],
    "Londrina": [-23.3045, -51.1696],
    "Maringá": [-23.4200, -51.9333],\
    "Caxias do Sul": [-29.1678, -51.1794],\
    "Pelotas": [-31.7649, -52.3371],\
    "Joinville": [-26.3045, -48.8487],\
}

casos_localidade = spark_df.groupBy("localidade").count()
casos_localidade_pd = casos_localidade.toPandas()

casos_localidade_pd['latitude'] = casos_localidade_pd['localidade'].map(lambda x: coords.get(x, [None, None])[0])
casos_localidade_pd['longitude'] = casos_localidade_pd['localidade'].map(lambda x: coords.get(x, [None, None])[1])

casos_localidade_pd = casos_localidade_pd.dropna(subset=['latitude', 'longitude'])

m = folium.Map(location=[-15.77972, -47.92972], zoom_start=4)

for idx, row in casos_localidade_pd.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(5, row['count']**0.5),
        popup=f"{row['localidade']}: {row['count']} casos",
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(m)

m.save("/app/output/mapa_casos_localidade.html") # Salva o mapa como HTML
# m # Removido, pois não há widget de exibição

# MODELOS DE CLASSIFICAÇÃO

# Vamos implementar três modelos de classificação para prever se um paciente tem IST
# baseado em suas características. Compararemos Logistic Regression, Random Forest e
# Naive Bayes para determinar qual tem melhor desempenho em nossa base de dados.

# Preparamos os dados para o modelo de classificação, selecionando as features e a variável alvo (label).
# Em seguida, dividimos o conjunto de dados em treino (70%) e teste (30%) para validar o modelo posteriormente.


# Preparar dados para classificação
classification_df = encoded_df.select("features", "tem_ist").withColumnRenamed("tem_ist", "label")

# Dividir em treino e teste
train_data, test_data = classification_df.randomSplit([0.7, 0.3], seed=42)

# Avaliadores do Spark
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Treinar e fazer previsões com o modelo Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# Cálculo das métricas usando as previsões do modelo Random Forest
acc = evaluator_acc.evaluate(rf_predictions)
precision = evaluator_precision.evaluate(rf_predictions)
recall = evaluator_recall.evaluate(rf_predictions)
f1 = evaluator_f1.evaluate(rf_predictions)

print(f"Acurácia: {acc:.4f}")
print(f"Precisão: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Matriz de Confusão (Convertendo para Pandas)
y_true = rf_predictions.select('label').toPandas()
y_pred = rf_predictions.select('prediction').toPandas()

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.savefig("/app/output/matriz_confusao.png") # Salva o gráfico
# plt.show() # Removido, pois não há widget de exibição


# Preparação dos dados para cálculo da Curva ROC e da métrica AUC:
# Extraímos as colunas de probabilidades previstas (onde o índice 1 corresponde à classe positiva) e os rótulos reais,
# convertendo o DataFrame Spark para Pandas, pois as funções de avaliação ROC e AUC do sklearn trabalham com arrays Pandas/NumPy.

# Calculamos a AUC (Área sob a Curva ROC), que quantifica a capacidade do modelo em distinguir entre as classes positivas e negativas.
# Um valor próximo de 1 indica ótimo desempenho, enquanto 0.5 indica desempenho aleatório.

# Geramos os valores de taxa de falsos positivos (FPR), taxa de verdadeiros positivos (TPR) e os limiares para a curva ROC.

# Por fim, plotamos a Curva ROC, que ilustra o trade-off entre sensibilidade (TPR) e especificidade (1-FPR) do modelo,
# facilitando a visualização da qualidade do classificador em diferentes limiares de decisão.


#Curva ROC e AUC

# Preparar dados para AUC e Curva ROC
# Replace 'predictions' with the actual variable holding the predictions, e.g., rf_predictions
preds = rf_predictions.select('probability', 'label').toPandas()
probs = preds['probability'].apply(lambda x: x[1])
labels = preds['label']

# Calcular AUC
auc = roc_auc_score(labels, probs)
print(f"AUC: {auc:.4f}")

# Curva ROC
fpr, tpr, thresholds = roc_curve(labels, probs)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig("/app/output/curva_roc.png") # Salva o gráfico
# plt.show() # Removido, pois não há widget de exibição

# Treinamento e avaliação de três modelos de classificação diferentes:

# 1. Regressão Logística:
# Modelo linear que estima a probabilidade da classe positiva a partir das variáveis preditoras.
# É simples e eficiente para problemas binários.

# 2. Random Forest:
# Ensemble de árvores de decisão que melhora a performance combinando múltiplas árvores.
# É robusto a overfitting e capaz de capturar relações não lineares nos dados.

# 3. Naive Bayes:
# Baseado no teorema de Bayes com a suposição de independência entre as variáveis preditoras.
# É rápido e funciona bem com dados categóricos e alta dimensionalidade.

# Para cada modelo, realizamos o treinamento com o conjunto de dados de treino e predições no conjunto de teste.

# Em seguida, avaliamos a acurácia de cada modelo, que indica a proporção de classificações corretas.
# Essa métrica serve para comparar rapidamente o desempenho dos modelos entre si.

# Por fim, imprimimos as acurácias para facilitar a análise de qual modelo apresenta melhor desempenho neste problema.


# Regressão Logística
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10)
rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# Naive Bayes
nb = NaiveBayes(featuresCol="features", labelCol="label")
nb_model = nb.fit(train_data)
nb_predictions = nb_model.transform(test_data)

# Avaliação dos modelos
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

lr_accuracy = evaluator.evaluate(lr_predictions, {evaluator.metricName: "accuracy"})
rf_accuracy = evaluator.evaluate(rf_predictions, {evaluator.metricName: "accuracy"})
nb_accuracy = evaluator.evaluate(nb_predictions, {evaluator.metricName: "accuracy"})

print("\nAcurácia dos Modelos de Classificação:")
print(f"Regressão Logística: {lr_accuracy:.4f}")
print(f"Random Forest: {rf_accuracy:.4f}")
print(f"Naive Bayes: {nb_accuracy:.4f}")

# Métricas detalhadas para o melhor modelo
print("\nRelatório de Classificação para Random Forest (melhor modelo):")
rf_predictions_pd = rf_predictions.select("label", "prediction").toPandas()
print(classification_report(rf_predictions_pd["label"], rf_predictions_pd["prediction"]))

"""
Comparação dos Modelos de Classificação:
O Random Forest obteve a melhor acurácia, seguido pela Regressão Logística e depois
Naive Bayes. Isso era esperado, pois Random Forests geralmente performam bem com
dados tabulares e podem capturar relações não-lineares. A Regressão Logística, sendo
um modelo linear, tem performance um pouco inferior. O Naive Bayes, que assume
independência entre features, teve o pior desempenho, possivelmente porque nossas
features não são completamente independentes.

Escolha do Melhor Modelo:
O Random Forest foi escolhido como melhor modelo devido à sua maior acurácia e
capacidade de lidar bem com nossos dados. Além disso, ele fornece importância das
features, o que é valioso para entender quais fatores mais influenciam a presença
de ISTs.
"""

# MODELOS DE CLUSTERIZAÇÃO

# A clusterização nos permite descobrir grupos naturais nos dados sem usar a variável
# alvo. Usaremos K-means, um algoritmo popular de clusterização, para identificar
# padrões e agrupamentos nos dados de ISTs.

# Preparamos os dados para o modelo de clusterização, selecionando as features.
# Em seguida, treinamos o modelo K-means com o número de clusters (k) e fazemos predições.

# Preparar dados para clusterização (usando as mesmas features do modelo de classificação)
clustering_df = encoded_df.select("features")

# Avaliar o custo para diferentes valores de K (método do cotovelo)
# Isso ajuda a escolher o número ideal de clusters
custos = []
for k in range(2, 8):
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model_kmeans = kmeans.fit(clustering_df)
    cost = model_kmeans.summary.trainingCost
    custos.append(cost)
    print(f"Custo para k={k}: {cost}")

# Plotar o método do cotovelo
plt.figure(figsize=(8,6))
plt.plot(range(2, 8), custos, marker='o', color='blue')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Custo (WCSS)')
plt.title('Método do Cotovelo para K-Means')
plt.grid(True)
plt.savefig("/app/output/metodo_cotovelo.png") # Salva o gráfico
# plt.show() # Removido, pois não há widget de exibição

# Baseado no método do cotovelo (assumindo que você analisaria o gráfico e escolheria um K)
# Vamos escolher k=4 como exemplo para continuar.
k_ideal = 4
kmeans_final = KMeans(featuresCol="features", k=k_ideal, seed=42)
model_kmeans_final = kmeans_final.fit(clustering_df)
clusters = model_kmeans_final.transform(clustering_df)

# Visualizar os clusters (ex: usando PCA para 2D)
# Isso ajuda a entender como os dados foram agrupados.

# Reduzir dimensionalidade com PCA para visualização
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(clusters)
pca_data = pca_model.transform(clusters).select("pca_features", "prediction")

# Converter para Pandas para plotagem
pca_data_pd = pca_data.toPandas()
pca_data_pd['x'] = pca_data_pd['pca_features'].apply(lambda v: v[0])
pca_data_pd['y'] = pca_data_pd['pca_features'].apply(lambda v: v[1])

plt.figure(figsize=(10,8))
sns.scatterplot(data=pca_data_pd, x='x', y='y', hue='prediction', palette='viridis', s=50, alpha=0.7)
plt.title(f'Visualização dos Clusters K-Means (K={k_ideal}) com PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')
plt.grid(True)
plt.savefig("/app/output/clusters_kmeans.png") # Salva o gráfico
# plt.show() # Removido, pois não há widget de exibição

# Parar a SparkSession ao final do notebook
spark.stop()